# Lecture 3: Linear Classifiers

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 3, with material from Stanford CS231n 2026, Lectures 2–3.*

The **linear classifier** is the building block of neural networks: every fully connected layer is one, and so is the last layer of every CNN.

**What you'll learn**
1. What a linear classifier computes: $f(x, W) = Wx + b$.
2. Three ways to picture it: **algebraic**, **visual** (templates), **geometric** (lines).
3. **Loss functions** that measure how good $W$ is: the **multiclass SVM** loss and the **softmax** (cross-entropy) loss.
4. **Regularization**: preferring simpler weights.

(How to *find* good weights is Lecture 4. Here we'll use a short training loop and explain it properly next time.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

## 1. The parametric approach

Nearest neighbor kept *all* the training data. A **parametric** model instead learns a fixed set of numbers, the **weights** $W$, and then throws the training data away:

$$f(x, W) = Wx + b$$

* $x$: the image, stretched into a column (a 32×32×3 CIFAR image is 3,072 numbers).
* $W$: one **row per class** (10 × 3,072 for CIFAR-10).
* $b$: one **bias** per class.
* The result is **10 scores**, one per class. The highest score wins.

### A tiny example
From the lecture: a 4-pixel image and 3 classes (cat, dog, ship). Each class score is a **weighted sum of the pixels**, plus that class's bias.

In [ ]:
x = np.array([56, 231, 24, 2])               # the image's 4 pixels, stretched into a column

W = np.array([[0.2, -0.5, 0.1, 2.0],         # cat weights
              [1.5, 1.3, 2.1, 0.0],          # dog weights
              [0.0, 0.25, 0.2, -0.3]])       # ship weights
b = np.array([1.1, 3.2, -1.2])

scores = W @ x + b
for name, score in zip(["cat", "dog", "ship"], scores):
    print(f"{name:4s} score: {score:8.2f}")

Let's check the cat score by hand: $0.2\cdot56 - 0.5\cdot231 + 0.1\cdot24 + 2.0\cdot2 + 1.1 = -96.8$. ✔

(The slide shows 61.95 for ship: that's the score **without** the bias. With the bias, $61.95 - 1.2 = 60.75$.)

This classifier thinks the image is a **dog**, by a lot. If the image is actually a cat, that's a bad set of weights. We'll need a way to measure *how* bad: a **loss function** (Section 4).

**The bias trick:** add a constant 1 to the end of $x$, and put $b$ in as an extra column of $W$. Then $f(x) = Wx$, one matrix instead of two.

In [ ]:
x_with_one = np.append(x, 1)                 # [56, 231, 24, 2, 1]
W_with_b = np.column_stack([W, b])           # W with b added as a last column
print("same scores with the bias trick:", W_with_b @ x_with_one)

## 2. Training a linear classifier on digits

To look at what linear classifiers learn, let's train one on our 8×8 digits. First, the data. Two standard preparation steps:
* **subtract the average training image**, so each pixel says "more or less ink than usual";
* **add a 1 to each image** (the bias trick).

In [ ]:
digits = load_digits()
images = digits.images / 16.0          # scale pixels to 0..1
labels = digits.target

rng = np.random.default_rng(0)
order = rng.permutation(len(images))
train_idx, val_idx, test_idx = order[:1000], order[1000:1300], order[1300:]

mean_image = images[train_idx].mean(axis=0)

def prepare(idx):
    flat = (images[idx] - mean_image).reshape(len(idx), 64)
    return np.column_stack([flat, np.ones(len(idx))])      # 64 pixels + a constant 1

X_train, y_train = prepare(train_idx), labels[train_idx]
X_val, y_val = prepare(val_idx), labels[val_idx]
X_test, y_test = prepare(test_idx), labels[test_idx]
print("each training image is now", X_train.shape[1], "numbers (64 pixels + 1)")

**A note on shapes:** the lecture writes one image as a *column*, $Wx$. In code we usually stack many images as *rows* of a matrix $X$ (one image per row) and compute all scores at once as `X @ W`, with $W$ of shape (65, 10). Same model, just transposed.

Here's a training function. Don't worry about how it works yet: it uses the **softmax loss** (Section 6) and **gradient descent** (Lecture 4). It repeatedly nudges $W$ in the direction that makes the loss smaller.

In [ ]:
def softmax_probabilities(scores):
    # Turn each row of scores into probabilities (Section 6 explains this).
    scores = scores - scores.max(axis=1, keepdims=True)    # for numerical safety
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

def train_softmax(X, y, reg=0.001, learning_rate=0.5, steps=1000):
    W = np.zeros((X.shape[1], 10))
    for step in range(steps):
        probs = softmax_probabilities(X @ W)
        probs[np.arange(len(y)), y] -= 1                   # gradient of the loss w.r.t. the scores
        gradient = X.T @ probs / len(y) + 2 * reg * W      # gradient w.r.t. W
        W = W - learning_rate * gradient                   # step downhill
    return W

def accuracy(W, X, y):
    predictions = np.argmax(X @ W, axis=1)                 # the class with the highest score
    return np.mean(predictions == y)

W_learned = train_softmax(X_train, y_train)
print(f"validation accuracy: {accuracy(W_learned, X_val, y_val):.1%}")

## 3. Three ways to think about a linear classifier

### Viewpoint 1: algebraic
Each score is a **weighted sum of the pixels**. A positive weight means "ink here is evidence *for* this class"; negative means evidence *against*.

### Viewpoint 2: visual (templates)
Each class's weights have one number per pixel, so we can **reshape them back into an 8×8 image**: a **template** for that class. The score measures how well the image matches the template.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 1.8))
for digit in range(10):
    template = W_learned[:64, digit].reshape(8, 8)      # this class's 64 pixel weights
    axes[digit].imshow(template, cmap="RdBu_r")
    axes[digit].set_title(digit)
    axes[digit].axis("off")
plt.suptitle("learned templates (red = ink here supports this digit, blue = ink here argues against it)", fontsize=9)
plt.show()

Each template looks like a blurry version of its digit. On CIFAR-10 the templates are blurry objects too: the car template is reddish (lots of red cars), and the plane and ship templates are mostly blue (sky and water). So **a linear classifier uses the background**, which helps on average but fails on a red car in front of the sea.

**The big limitation: one template per class.** If a class looks different in different images (horses facing left *and* right), its single template has to **average** them. The CIFAR-10 horse template famously has **two heads**. We can create the same effect: train on our digits **plus their mirror images**, as if digits could face either way.

In [ ]:
mirrored = images[:, :, ::-1]                      # flip every image left-to-right
X_mirror = (mirrored[train_idx] - mean_image).reshape(1000, 64)
X_mirror = np.column_stack([X_mirror, np.ones(1000)])

X_both = np.vstack([X_train, X_mirror])            # original + mirrored training images
y_both = np.concatenate([y_train, y_train])
W_both = train_softmax(X_both, y_both)

fig, axes = plt.subplots(2, 4, figsize=(6, 3.3))
for col, digit in enumerate([2, 3, 4, 7]):
    axes[0, col].imshow(W_learned[:64, digit].reshape(8, 8), cmap="RdBu_r"); axes[0, col].set_title(f"{digit}: normal", fontsize=8)
    axes[1, col].imshow(W_both[:64, digit].reshape(8, 8), cmap="RdBu_r"); axes[1, col].set_title(f"{digit}: + mirrored", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.show()
print(f"accuracy on normal validation digits: trained normally {accuracy(W_learned, X_val, y_val):.1%}, "
      f"trained with mirrored copies {accuracy(W_both, X_val, y_val):.1%}")

**What we see:** with mirrored copies in the data, each template becomes a symmetric blend of both orientations (the digits' "two-headed horse"), and accuracy drops. One template can't represent "this shape OR its mirror image". A neural network with a hidden layer can (Lecture 5).

### Viewpoint 3: geometric
Think of each image as a **point** in space (one axis per pixel). Each class score is a linear function of that point, so the boundary between two classes is a **straight line** (a hyperplane in high dimensions). In 2-D we can draw it:

In [ ]:
rng = np.random.default_rng(1)
centers = [[-2, 0], [2, 1], [0, 2.5]]
points = np.vstack([rng.normal(center, 0.7, size=(50, 2)) for center in centers])
point_labels = np.repeat([0, 1, 2], 50)
points_with_one = np.column_stack([points, np.ones(150)])

def train_softmax_small(X, y, n_classes, steps=2000):
    W = np.zeros((X.shape[1], n_classes))
    for step in range(steps):
        probs = softmax_probabilities(X @ W)
        probs[np.arange(len(y)), y] -= 1
        W = W - 0.5 * X.T @ probs / len(y)
    return W

W_2d = train_softmax_small(points_with_one, point_labels, 3)

grid_x, grid_y = np.meshgrid(np.linspace(-4.5, 4.5, 200), np.linspace(-2.5, 5, 200))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel(), np.ones(grid_x.size)])
regions = np.argmax(grid @ W_2d, axis=1).reshape(grid_x.shape)

plt.figure(figsize=(5, 4))
plt.contourf(grid_x, grid_y, regions, alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5], cmap="brg")
plt.scatter(points[:, 0], points[:, 1], c=point_labels, cmap="brg", edgecolors="k", s=20)
plt.title("a linear classifier cuts the plane with straight lines")
plt.show()

**Hard cases for a linear classifier:** some patterns can't be separated by straight lines at all, e.g., **XOR** (first and third quadrants vs. second and fourth), a **ring** (inside a band vs. outside), or a class with **several separate blobs**. Before deep learning, people fixed this by hand-designing **features** that make the data separable (Lecture 5 shows this and the neural-network solution).

## 4. Loss functions: how good is W?

A **loss function** turns "how wrong are the scores?" into one number. For a dataset of $N$ examples, the loss is the average of the per-example losses:

$$L = \frac{1}{N}\sum_{i=1}^{N}L_i\big(f(x_i, W),\ y_i\big)$$

The lecture's example: three training images and the scores some $W$ gave them (correct class in bold):

| | cat image | car image | frog image |
|---|---|---|---|
| cat score | **3.2** | 1.3 | 2.2 |
| car score | 5.1 | **4.9** | 2.5 |
| frog score | −1.7 | 2.0 | **−3.1** |

### The multiclass SVM loss
**Idea:** the correct class should score higher than every other class **by at least 1** (the "margin"). Each wrong class that gets too close adds a penalty:

$$L_i = \sum_{j \ne y_i}\max\left(0,\ s_j - s_{y_i} + 1\right)$$

By hand, for the **cat** image (correct score 3.2):
* car: $\max(0,\ 5.1 - 3.2 + 1) = 2.9$
* frog: $\max(0,\ -1.7 - 3.2 + 1) = 0$
* total: $L = 2.9$

In [ ]:
scores = np.array([[3.2, 5.1, -1.7],     # cat image: scores for (cat, car, frog)
                   [1.3, 4.9, 2.0],      # car image
                   [2.2, 2.5, -3.1]])    # frog image
correct = [0, 1, 2]                      # the correct class of each image

def svm_loss(score_row, correct_class):
    loss = 0
    for j in range(len(score_row)):
        if j != correct_class:
            loss += max(0, score_row[j] - score_row[correct_class] + 1)
    return loss

losses = [svm_loss(scores[i], correct[i]) for i in range(3)]
for name, loss in zip(["cat", "car", "frog"], losses):
    print(f"{name} image: loss {loss:.1f}")
print(f"average loss: {np.mean(losses):.2f}")

That matches the lecture: 2.9, 0, 12.9, average 5.27.

**Questions from the lecture:**
1. **What's the smallest and largest possible loss?** 0 (every margin satisfied) and infinity.
2. **At the start of training, $W$ is tiny, so all scores are about 0. What's the loss?** Each of the $C - 1$ wrong classes adds $\max(0, 0 - 0 + 1) = 1$, so about $C - 1$. **A great sanity check** for your code:

In [ ]:
rng = np.random.default_rng(0)
tiny_scores = rng.normal(0, 0.001, size=(500, 10))        # 10 classes, scores all near 0
fake_labels = rng.integers(0, 10, 500)
average = np.mean([svm_loss(tiny_scores[i], fake_labels[i]) for i in range(500)])
print(f"average SVM loss with tiny scores: {average:.3f}  (expected C - 1 = 9)")

3. **If $W$ gives zero loss, is it the only one?** No: $2W$ doubles every score, so every margin is still satisfied and the loss is still 0. So the loss alone can't tell us which $W$ to prefer. That's where **regularization** comes in.

## 5. Regularization

We add a second term that prefers "simpler" weights:

$$L(W) = \underbrace{\frac{1}{N}\sum_i L_i}_{\text{fit the training data}} + \underbrace{\lambda\,R(W)}_{\text{keep W simple}}$$

* **L2 regularization:** $R(W) = \sum W^2$ (the most common)
* **L1 regularization:** $R(W) = \sum |W|$
* $\lambda$ is a **hyperparameter** that sets the balance.

**Why?** To express preferences between equally good weights, and to avoid **overfitting** (fitting the noise in the training data instead of the real pattern). The lecture's example: for $x = [1, 1, 1, 1]$, both $w_1 = [1, 0, 0, 0]$ and $w_2 = [0.25, 0.25, 0.25, 0.25]$ give the same score, 1. Which does L2 prefer?

In [ ]:
x4 = np.array([1, 1, 1, 1])
w1 = np.array([1, 0, 0, 0])
w2 = np.array([0.25, 0.25, 0.25, 0.25])
print("scores:", w1 @ x4, "and", w2 @ x4)
print("L2 penalty:", np.sum(w1**2), "vs", np.sum(w2**2), " -> L2 prefers w2 (spread out)")
print("L1 penalty:", np.sum(np.abs(w1)), "vs", np.sum(np.abs(w2)), " -> a tie")

**L2 likes to spread the weights out**, so the classifier uses all the evidence instead of relying on one pixel. **L1 tends to make many weights exactly zero** ("sparse").

Regularization matters most when there's **little training data** and we train for a **long** time (so the model has a chance to overfit). Train on just 150 images, for 20,000 steps, with different $\lambda$:

In [ ]:
print(f"{'lambda':>7s} {'train acc':>10s} {'val acc':>8s}")
for lam in [0.0, 0.001, 0.01, 0.1]:
    W_small = train_softmax(X_train[:150], y_train[:150], reg=lam, learning_rate=1.0, steps=20000)
    print(f"{lam:7g} {accuracy(W_small, X_train[:150], y_train[:150]):10.1%} {accuracy(W_small, X_val, y_val):8.1%}")

**What we see:** as $\lambda$ grows, training accuracy drops (the model isn't allowed to fit every training image). Validation accuracy is best at a **small** $\lambda$: no regularization overfits a little, and too much hurts both. (If you train for only a few steps, starting from $W = 0$, the weights never get large anyway, and the benefit of regularization mostly disappears. Try `steps=1000`.)

## 6. The softmax classifier

The SVM treats scores as just numbers. **Softmax turns scores into probabilities**:
1. **Exponentiate** each score (makes everything positive).
2. **Normalize** (divide by the total, so they sum to 1).
3. The loss is the **negative log of the correct class's probability**: $L_i = -\log P(\text{correct class})$.

Step by step for the cat image (scores: cat 3.2, car 5.1, frog −1.7):

In [ ]:
cat_scores = np.array([3.2, 5.1, -1.7])

step1 = np.exp(cat_scores)
print("1. exponentiate:", step1.round(2))

step2 = step1 / step1.sum()
print("2. normalize:   ", step2.round(2), "(sums to", step2.sum().round(2), ")")

loss = -np.log(step2[0])                   # the correct class is cat (index 0)
print(f"3. loss = -log({step2[0]:.2f}) = {loss:.2f}")

That matches the lecture (24.5, 164.0, 0.18 → 0.13, 0.87, 0.00 → loss 2.04).

**Why the log?** If the correct class gets probability 1, the loss is 0; as that probability goes to 0, the loss grows without limit. Minimizing it means **making the correct labels as probable as possible** (maximum likelihood). It's also called the **cross-entropy loss**.

**Sanity check at the start of training:** all scores about equal, so every class gets probability $1/C$, and the loss is $-\log(1/C) = \log C$. For 10 classes, about 2.3:

In [ ]:
W_zero = np.zeros((65, 10))
probs = softmax_probabilities(X_train @ W_zero)
start_loss = -np.mean(np.log(probs[np.arange(len(y_train)), y_train]))
print(f"softmax loss with W = 0: {start_loss:.4f}   log(10) = {np.log(10):.4f}")

**Computing softmax safely:** $e^{1000}$ overflows to infinity. Since softmax gives the same answer if you subtract the same number from all the scores, we subtract the largest score first. That's the `scores.max()` line in `softmax_probabilities`.

In [ ]:
big_scores = np.array([1000.0, 1001.0, 999.0])
safe = np.exp(big_scores - big_scores.max())
print("safe softmax of [1000, 1001, 999]:", (safe / safe.sum()).round(3))

## 7. SVM vs. softmax

Both are linear classifiers; they differ only in the loss. The difference shows when scores are already good. Raise the correct score further, from 10 to 20:

In [ ]:
def softmax_loss(score_row, correct_class):
    shifted = score_row - score_row.max()
    probs = np.exp(shifted) / np.exp(shifted).sum()
    return -np.log(probs[correct_class])

for correct_score in [10, 20]:
    row = np.array([correct_score, -2.0, 3.0])          # class 0 is correct
    print(f"correct score {correct_score}: SVM loss {svm_loss(row, 0):.1f},   softmax loss {softmax_loss(row, 0):.2e}")

* **SVM:** once the margins are satisfied, the loss is 0 and it stops caring.
* **Softmax:** the loss keeps shrinking; it always wants the correct class to be even more probable.

In practice they reach similar accuracy. Softmax is more common because it gives **probabilities**.

## 8. Final test accuracy

Choose $\lambda$ on the validation set, then evaluate on the test set once:

In [ ]:
best_lambda, best_val = None, 0
for lam in [0.0001, 0.001, 0.01]:
    W_try = train_softmax(X_train, y_train, reg=lam)
    val_acc = accuracy(W_try, X_val, y_val)
    print(f"lambda {lam}: validation {val_acc:.1%}")
    if val_acc > best_val:
        best_lambda, best_val = lam, val_acc

W_final = train_softmax(X_train, y_train, reg=best_lambda)
print(f"\nchosen lambda = {best_lambda};  test accuracy: {accuracy(W_final, X_test, y_test):.1%}")

About 95–96% is roughly the best a *linear* classifier can do on these digits. A neural network does better (Lecture 5).

## Summary
* A linear classifier computes **scores** $f(x) = Wx + b$: one row of $W$ per class.
* Three views: **weighted sums of pixels**, **one template per class**, **straight-line boundaries**.
* A **loss function** measures how bad the scores are:
  * **SVM:** $\sum_{j\ne y}\max(0, s_j - s_y + 1)$; starts near $C - 1$.
  * **Softmax:** $-\log P(\text{correct})$; starts near $\log C$.
* **Regularization** $\lambda R(W)$ prefers simpler weights and reduces overfitting.

### Check your understanding
1. For CIFAR-10, what is the shape of $W$, and what does one row represent?
2. Why does the horse template have two heads?
3. A 10-class softmax classifier starts with loss 8.5. Is something wrong?
4. Why doesn't the SVM loss alone decide between $W$ and $2W$?

<details><summary>Answers</summary>

1. 10 × 3,072 (plus the bias). One row is the template for one class.
2. Horses face both left and right; one template has to average both.
3. Probably: the starting loss should be near log 10 ≈ 2.3.
4. Doubling $W$ doubles every margin, so satisfied margins stay satisfied and the loss stays 0. Regularization breaks the tie.
</details>

**Next:** Lecture 4, optimization (how `train_softmax` actually works).